# 05 - CatBoost Model Development

This notebook is the active modeling laboratory for CatBoost. The baseline notebook established a strong point-accuracy benchmark; the weather EDA notebook established what exogenous information is available. Here we treat CatBoost as a rolling adaptation problem: the experiment compares feature sets, target modes, retraining cadence, retuning cadence, training-window length, and recency weighting as one operational policy.

<p style="color:#b00020"><strong>Interpretation guide.</strong> A policy is the unit of comparison. Each policy says how often CatBoost retunes hyperparameters, how often it refits model weights inside the validation block, how much history it uses, and how much it upweights recent observations. The notebook rolls forward block by block, scores each policy on future origins, then lets the next block train and tune with the newly observed data included.</p>

> **Runtime note.** The default cells still run full Optuna tuning. The policy grid is deliberately compact rather than fully Cartesian: it compares a few interpretable adaptation regimes without exploding into every possible combination of cadence, window, and weighting.


## 1. Setup And Modeling Contract

The modeling problem is not just selecting a CatBoost feature matrix. Danish power prices drift, and the available tabular features only partially explain that drift. The central experiment is therefore a meta-optimization over learning policies: should the model keep all history, downweight old history, use a hard recent window, refit weekly, refit monthly, and retune every quarter?

For each validation block, CatBoost tunes candidates using only data available before the block, replays the chosen hyperparameters through the block on a retraining schedule, and scores every forecast origin. When the next block starts, the previous block's actuals are available for both training and tuning. This is the workflow a production learner would follow, minus the final publishing step.


In [1]:
from __future__ import annotations

import json
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dkenergy_forecast.backtesting.horizons import make_daily_origins
from dkenergy_forecast.evaluation.summary import model_score_table
from dkenergy_forecast.io import load_price_panel
from dkenergy_forecast.features.price_features import (
    WEIGHTED_MEDIAN_BASELINE_COLUMN,
    add_weighted_median_baseline_feature,
    build_price_experiment_frame,
)
from dkenergy_forecast.tuning import (
    CatBoostValidationConfig,
    RecencyWeightSpec,
    build_candidate_grid,
    run_baseline_comparator,
    run_catboost_validation,
    write_catboost_validation_artifacts,
)
from dkenergy_forecast.tuning.catboost_validation import (
    make_retune_month_blocks,
    outer_month_score_table,
    outer_origins_for_months,
    prepare_nested_frame,
    validation_block_label,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [2]:
PANEL_PATH = PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.parquet"
QA_PATH = PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.qa.json"
PRICE_FRAME_PATH = PROJECT_ROOT / "data" / "features" / "price_experiment_frame_policy_v1.parquet"
WEATHER_FRAME_PATH = PROJECT_ROOT / "data" / "features" / "weather_experiment_frame_backtest.parquet"
RESULT_ROOT = PROJECT_ROOT / "results" / "notebook_catboost_development_v1"
PRICE_RESULT_DIR = RESULT_ROOT / "price_policy_validation"
WEATHER_RESULT_DIR = RESULT_ROOT / "weather_policy_validation"

PRICE_FRAME_MONTHS = tuple(str(period) for period in pd.period_range("2012-01", "2026-06", freq="M"))
PRICE_BACKTEST_MONTHS = tuple(str(period) for period in pd.period_range("2013-04", "2026-06", freq="M"))
WEATHER_BACKTEST_MONTHS = tuple(str(period) for period in pd.period_range("2025-04", "2026-06", freq="M"))
RUN_FULL_TUNING = True
N_TRIALS = 8

BASE_TUNING_CONFIG = dict(
    n_trials=N_TRIALS,
    min_train_rows=1000,
    max_iterations=900,
    early_stopping_rounds=80,
    eval_origin_count=14,  # latest training origins reserved as CatBoost's fit-time eval_set
    random_seed=42,
    has_time=True,
    replay_all_candidates=True,
)

PRICE_POLICY_GRID = [
    dict(
        policy_label="q3_r7_full_hl180",
        retune_every_months=3,
        retrain_every_days=7,
        training_origin_days=None,
        recency=RecencyWeightSpec("exp_hl180", half_life_days=180),
        rationale="quarterly retune, weekly refit, all history with moderate recency bias",
    ),
    dict(
        policy_label="q3_r7_full_hl720",
        retune_every_months=3,
        retrain_every_days=7,
        training_origin_days=None,
        recency=RecencyWeightSpec("exp_hl720", half_life_days=720),
        rationale="quarterly retune, weekly refit, all history with weak recency bias",
    ),
    dict(
        policy_label="q3_r32_full_unweighted",
        retune_every_months=3,
        retrain_every_days=32,
        training_origin_days=None,
        recency=RecencyWeightSpec("unweighted"),
        rationale="quarterly retune, monthly-ish refit, all history as an anchor",
    ),
    dict(
        policy_label="q3_r32_full_hl180",
        retune_every_months=3,
        retrain_every_days=32,
        training_origin_days=None,
        recency=RecencyWeightSpec("exp_hl180", half_life_days=180),
        rationale="default middle path: all history plus moderate recency weighting",
    ),
    dict(
        policy_label="q3_r32_full_hl720",
        retune_every_months=3,
        retrain_every_days=32,
        training_origin_days=None,
        recency=RecencyWeightSpec("exp_hl720", half_life_days=720),
        rationale="conservative path: all history plus weak recency weighting",
    ),
    dict(
        policy_label="q3_r32_full_hl90_floor20",
        retune_every_months=3,
        retrain_every_days=32,
        training_origin_days=None,
        recency=RecencyWeightSpec("exp_hl90_floor20", half_life_days=90, floor=0.20),
        rationale="aggressive recency bias while keeping a 20% old-data anchor",
    ),
    dict(
        policy_label="q3_r32_365d_unweighted",
        retune_every_months=3,
        retrain_every_days=32,
        training_origin_days=365,
        recency=RecencyWeightSpec("unweighted"),
        rationale="hard one-year memory without extra recency weighting",
    ),
    dict(
        policy_label="q3_r32_365d_hl180",
        retune_every_months=3,
        retrain_every_days=32,
        training_origin_days=365,
        recency=RecencyWeightSpec("exp_hl180", half_life_days=180),
        rationale="one-year memory plus gentle weighting toward recent observations",
    ),
]

WEATHER_POLICY_GRID = [
    policy
    for policy in PRICE_POLICY_GRID
    if policy["policy_label"] not in {"q3_r7_full_hl720", "q3_r32_full_hl720"}
]

PRICE_FEATURE_SETS = [
    "price_full_engineered",
    "price_baseline_calendar",
    "price_lags_calendar",
]
WEATHER_FEATURE_SETS = [
    "price_full_engineered",
    "ensemble",
    "all_weather",
    "all_weather_plus_ensemble",
]
TARGET_MODES = ["direct", "residual_baseline"]
SEARCH_PROFILES = ["conservative"]

def policy_overview_rows(policies, *, experiment, backtest_months):
    return [
        {
            "experiment": experiment,
            "policy_label": policy["policy_label"],
            "retune_every_months": policy["retune_every_months"],
            "retrain_every_days": policy["retrain_every_days"],
            "training_origin_days": policy["training_origin_days"],
            "recency_label": policy["recency"].label,
            "sample_weight_half_life_days": policy["recency"].half_life_days,
            "sample_weight_floor": policy["recency"].floor,
            "validation_blocks": make_retune_month_blocks(
                backtest_months,
                retune_every_months=policy["retune_every_months"],
            ),
            "rationale": policy["rationale"],
        }
        for policy in policies
    ]


policy_overview = pd.DataFrame(
    policy_overview_rows(PRICE_POLICY_GRID, experiment="price", backtest_months=PRICE_BACKTEST_MONTHS)
    + policy_overview_rows(WEATHER_POLICY_GRID, experiment="weather", backtest_months=WEATHER_BACKTEST_MONTHS)
)
policy_overview


,experiment,policy_label,retune_every_months,retrain_every_days,training_origin_days,recency_label,sample_weight_half_life_days,sample_weight_floor,validation_blocks,rationale
0,price,q3_r7_full_hl180,3,7,NaN,exp_hl180,180.0,NaN,"[(2013-04, 2013-05, 2013-06), (2013-07, 2013-0...","quarterly retune, weekly refit, all history wi..."
1,price,q3_r7_full_hl720,3,7,NaN,exp_hl720,720.0,NaN,"[(2013-04, 2013-05, 2013-06), (2013-07, 2013-0...","quarterly retune, weekly refit, all history wi..."
2,price,q3_r32_full_unweighted,3,32,NaN,unweighted,NaN,NaN,"[(2013-04, 2013-05, 2013-06), (2013-07, 2013-0...","quarterly retune, monthly-ish refit, all histo..."
3,price,q3_r32_full_hl180,3,32,NaN,exp_hl180,180.0,NaN,"[(2013-04, 2013-05, 2013-06), (2013-07, 2013-0...",default middle path: all history plus moderate...
4,price,q3_r32_full_hl720,3,32,NaN,exp_hl720,720.0,NaN,"[(2013-04, 2013-05, 2013-06), (2013-07, 2013-0...",conservative path: all history plus weak recen...
5,price,q3_r32_full_hl90_floor20,3,32,NaN,exp_hl90_floor20,90.0,0.2,"[(2013-04, 2013-05, 2013-06), (2013-07, 2013-0...",aggressive recency bias while keeping a 20% ol...
6,price,q3_r32_365d_unweighted,3,32,365.0,unweighted,NaN,NaN,"[(2013-04, 2013-05, 2013-06), (2013-07, 2013-0...",hard one-year memory without extra recency wei...
7,price,q3_r32_365d_hl180,3,32,365.0,exp_hl180,180.0,NaN,"[(2013-04, 2013-05, 2013-06), (2013-07, 2013-0...",one-year memory plus gentle weighting toward r...
8,weather,q3_r7_full_hl180,3,7,NaN,exp_hl180,180.0,NaN,"[(2025-04, 2025-05, 2025-06), (2025-07, 2025-0...","quarterly retune, weekly refit, all history wi..."
9,weather,q3_r32_full_unweighted,3,32,NaN,unweighted,NaN,NaN,"[(2025-04, 2025-05, 2025-06), (2025-07, 2025-0...","quarterly retune, monthly-ish refit, all histo..."


## 2. Load Price And Weather Experiment Frames

The price-only policy experiment can use the long EDS price history. The weather experiment cannot: the cached Open-Meteo feature frame starts in mid-2024. We therefore split the notebook into two arenas. `PRICE_FRAME_MONTHS` is the price-only context frame, while `PRICE_BACKTEST_MONTHS` is the scored price arena. The first scored month starts later than the frame so the first retune has prior training rows and a prior tuning-validation window. `PRICE_BACKTEST_MONTHS` starts in 2013-04 after the 2012 context year and a Jan-Mar tuning window. `WEATHER_BACKTEST_MONTHS` starts in 2025-04 for the same reason: the cached weather frame has enough mid-2024 context and a Jan-Mar tuning window before the first scored weather block.


In [3]:
def month_span_origins(panel, months, *, at_hour_utc=10):
    start = pd.Period(months[0], freq="M").to_timestamp().tz_localize("UTC")
    end = (pd.Period(months[-1], freq="M") + 1).to_timestamp().tz_localize("UTC")
    return make_daily_origins(panel, start=start, end=end, at_hour_utc=at_hour_utc)


def build_or_load_price_policy_frame(panel, months):
    if PRICE_FRAME_PATH.exists():
        cached = pd.read_parquet(PRICE_FRAME_PATH)
        cached = prepare_nested_frame(cached)
        missing = sorted(set(months) - set(cached["outer_month"].dropna().unique()))
        if not missing:
            return cached
        print(f"Cached price policy frame is missing month(s) {missing}; rebuilding {PRICE_FRAME_PATH}.")

    origins = month_span_origins(panel, months)
    output = build_price_experiment_frame(panel, origins)
    PRICE_FRAME_PATH.parent.mkdir(parents=True, exist_ok=True)
    output.to_parquet(PRICE_FRAME_PATH, index=False)
    return prepare_nested_frame(output)


def require_months(frame, months, label):
    missing = sorted(set(months) - set(frame["outer_month"].dropna().unique()))
    if missing:
        raise ValueError(f"{label} is missing requested backtest month(s): {missing}")


def require_complete_baseline(frame, label):
    if frame[WEIGHTED_MEDIAN_BASELINE_COLUMN].isna().any():
        missing = int(frame[WEIGHTED_MEDIAN_BASELINE_COLUMN].isna().sum())
        raise ValueError(
            f"{WEIGHTED_MEDIAN_BASELINE_COLUMN} must be complete for residual CatBoost in {label}; missing_rows={missing}"
        )


if not WEATHER_FRAME_PATH.exists():
    raise FileNotFoundError(
        f"Missing {WEATHER_FRAME_PATH}. Build it with scripts/build_weather_backtest_frame.py before running this notebook."
    )

panel = load_price_panel(PANEL_PATH, QA_PATH if QA_PATH.exists() else None)
price_frame = build_or_load_price_policy_frame(panel, PRICE_FRAME_MONTHS)
weather_frame = pd.read_parquet(WEATHER_FRAME_PATH)

if WEIGHTED_MEDIAN_BASELINE_COLUMN not in weather_frame.columns:
    print(
        f"Cached weather experiment frame is missing {WEIGHTED_MEDIAN_BASELINE_COLUMN}; "
        "adding it with the package price-feature builder and updating the cached parquet."
    )
    weather_frame = add_weighted_median_baseline_feature(weather_frame, panel)
    weather_frame.to_parquet(WEATHER_FRAME_PATH, index=False)

weather_frame = prepare_nested_frame(weather_frame)
require_months(price_frame, PRICE_FRAME_MONTHS, "price policy frame")
require_months(weather_frame, WEATHER_BACKTEST_MONTHS, "weather experiment frame")
require_complete_baseline(price_frame, "price policy frame")
require_complete_baseline(weather_frame, "weather experiment frame")

price_outer_origins = outer_origins_for_months(price_frame, PRICE_BACKTEST_MONTHS)
weather_outer_origins = outer_origins_for_months(weather_frame, WEATHER_BACKTEST_MONTHS)

price_coverage = (
    price_frame.groupby("outer_month")
    .agg(
        origins=("forecast_origin_utc", "nunique"),
        rows=("y", "size"),
        first_origin=("forecast_origin_utc", "min"),
        last_origin=("forecast_origin_utc", "max"),
    )
    .loc[list(PRICE_BACKTEST_MONTHS)]
    .assign(experiment="price")
)
weather_coverage = (
    weather_frame.groupby("outer_month")
    .agg(
        origins=("forecast_origin_utc", "nunique"),
        rows=("y", "size"),
        first_origin=("forecast_origin_utc", "min"),
        last_origin=("forecast_origin_utc", "max"),
    )
    .loc[list(WEATHER_BACKTEST_MONTHS)]
    .assign(experiment="weather")
)
coverage_table = pd.concat([price_coverage, weather_coverage]).reset_index().rename(columns={"index": "outer_month"})


def first_block_preflight(frame, *, experiment, backtest_months, policy_grid):
    rows = []
    frame = prepare_nested_frame(frame)
    first_scored_month = backtest_months[0]
    for policy in policy_grid:
        first_block_months = make_retune_month_blocks(
            backtest_months,
            retune_every_months=policy["retune_every_months"],
        )[0]
        retune_at = pd.Timestamp(f"{first_scored_month}-01T00:00:00Z")
        tuning_start = retune_at - pd.DateOffset(months=policy["retune_every_months"])
        tuning_end = retune_at
        train_mask = (
            (frame["forecast_origin_utc"] < tuning_start)
            & (frame["ds_utc"] < tuning_start)
            & frame["y"].notna()
        )
        if policy["training_origin_days"] is not None:
            train_mask &= frame["forecast_origin_utc"] >= tuning_start - pd.Timedelta(days=policy["training_origin_days"])
        train_origins = frame.loc[train_mask, "forecast_origin_utc"].drop_duplicates().sort_values()
        train_origin_span_days = (
            (train_origins.max() - train_origins.min()) / pd.Timedelta(days=1)
            if len(train_origins) > 1
            else 0
        )
        tuning_mask = (
            (frame["forecast_origin_utc"] >= tuning_start)
            & (frame["forecast_origin_utc"] < tuning_end)
        )
        scored_mask = frame["outer_month"].isin(first_block_months)
        rows.append({
            "experiment": experiment,
            "policy_label": policy["policy_label"],
            "first_scored_block": validation_block_label(first_block_months),
            "retune_at_utc": retune_at,
            "tuning_start_utc": tuning_start,
            "tuning_end_utc": tuning_end,
            "train_rows_before_tuning_window": int(train_mask.sum()),
            "train_origins_before_tuning_window": int(len(train_origins)),
            "train_origin_span_days": float(train_origin_span_days),
            "tuning_validation_origins": int(frame.loc[tuning_mask, "forecast_origin_utc"].nunique()),
            "scored_origins_in_first_block": int(frame.loc[scored_mask, "forecast_origin_utc"].nunique()),
            "passes_min_train_rows": bool(train_mask.sum() >= BASE_TUNING_CONFIG["min_train_rows"]),
        })
    return pd.DataFrame(rows)


preflight_table = pd.concat(
    [
        first_block_preflight(
            price_frame,
            experiment="price",
            backtest_months=PRICE_BACKTEST_MONTHS,
            policy_grid=PRICE_POLICY_GRID,
        ),
        first_block_preflight(
            weather_frame,
            experiment="weather",
            backtest_months=WEATHER_BACKTEST_MONTHS,
            policy_grid=WEATHER_POLICY_GRID,
        ),
    ],
    ignore_index=True,
)


def require_preflight_context(table):
    missing_context = table[
        ~table["passes_min_train_rows"]
        | table["tuning_validation_origins"].eq(0)
        | table["scored_origins_in_first_block"].eq(0)
    ]
    if not missing_context.empty:
        columns = [
            "experiment",
            "policy_label",
            "train_rows_before_tuning_window",
            "train_origins_before_tuning_window",
            "train_origin_span_days",
            "tuning_validation_origins",
            "scored_origins_in_first_block",
        ]
        raise ValueError(
            "First scored block lacks enough tuning/training context:\n"
            + missing_context[columns].to_string(index=False)
        )


require_preflight_context(preflight_table)
display(coverage_table)
preflight_table


,outer_month,origins,rows,first_origin,last_origin,experiment
0,2013-04,30,1440,2013-04-01 10:00:00+00:00,2013-04-30 10:00:00+00:00,price
1,2013-05,31,1488,2013-05-01 10:00:00+00:00,2013-05-31 10:00:00+00:00,price
2,2013-06,30,1440,2013-06-01 10:00:00+00:00,2013-06-30 10:00:00+00:00,price
3,2013-07,31,1488,2013-07-01 10:00:00+00:00,2013-07-31 10:00:00+00:00,price
4,2013-08,31,1488,2013-08-01 10:00:00+00:00,2013-08-31 10:00:00+00:00,price
...,...,...,...,...,...,...
169,2026-02,28,1344,2026-02-01 10:00:00+00:00,2026-02-28 10:00:00+00:00,weather
170,2026-03,31,1486,2026-03-01 10:00:00+00:00,2026-03-31 10:00:00+00:00,weather
171,2026-04,30,1440,2026-04-01 10:00:00+00:00,2026-04-30 10:00:00+00:00,weather
172,2026-05,31,1488,2026-05-01 10:00:00+00:00,2026-05-31 10:00:00+00:00,weather


,experiment,policy_label,first_scored_block,retune_at_utc,tuning_start_utc,tuning_end_utc,train_rows_before_tuning_window,train_origins_before_tuning_window,train_origin_span_days,tuning_validation_origins,scored_origins_in_first_block,passes_min_train_rows
0,price,q3_r7_full_hl180,2013-04..2013-06,2013-04-01 00:00:00+00:00,2013-01-01 00:00:00+00:00,2013-04-01 00:00:00+00:00,17522,366,365.0,90,91,True
1,price,q3_r7_full_hl720,2013-04..2013-06,2013-04-01 00:00:00+00:00,2013-01-01 00:00:00+00:00,2013-04-01 00:00:00+00:00,17522,366,365.0,90,91,True
2,price,q3_r32_full_unweighted,2013-04..2013-06,2013-04-01 00:00:00+00:00,2013-01-01 00:00:00+00:00,2013-04-01 00:00:00+00:00,17522,366,365.0,90,91,True
3,price,q3_r32_full_hl180,2013-04..2013-06,2013-04-01 00:00:00+00:00,2013-01-01 00:00:00+00:00,2013-04-01 00:00:00+00:00,17522,366,365.0,90,91,True
4,price,q3_r32_full_hl720,2013-04..2013-06,2013-04-01 00:00:00+00:00,2013-01-01 00:00:00+00:00,2013-04-01 00:00:00+00:00,17522,366,365.0,90,91,True
5,price,q3_r32_full_hl90_floor20,2013-04..2013-06,2013-04-01 00:00:00+00:00,2013-01-01 00:00:00+00:00,2013-04-01 00:00:00+00:00,17522,366,365.0,90,91,True
6,price,q3_r32_365d_unweighted,2013-04..2013-06,2013-04-01 00:00:00+00:00,2013-01-01 00:00:00+00:00,2013-04-01 00:00:00+00:00,17474,365,364.0,90,91,True
7,price,q3_r32_365d_hl180,2013-04..2013-06,2013-04-01 00:00:00+00:00,2013-01-01 00:00:00+00:00,2013-04-01 00:00:00+00:00,17474,365,364.0,90,91,True
8,weather,q3_r7_full_hl180,2025-04..2025-06,2025-04-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,2025-04-01 00:00:00+00:00,8836,185,184.0,90,91,True
9,weather,q3_r32_full_unweighted,2025-04..2025-06,2025-04-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,2025-04-01 00:00:00+00:00,8836,185,184.0,90,91,True


<p style="color:#b00020"><strong>Interpretation.</strong> The long price arena tests adaptation policies across many market regimes. The weather arena answers a narrower question on the overlapping feature history: does weather add value once price-only policy behavior is understood? These two scores should not be collapsed as if they used the same historical span.</p>


## 3. Baselines On The Same Outer Origins

The production baselines are not a side note; they are the hurdle. A CatBoost model only matters if it beats these on the same outer origins, not on a convenient validation slice.


In [ ]:
price_baseline_predictions = run_baseline_comparator(
    panel,
    origins=price_outer_origins,
    min_train_days=365,
)
price_baseline_scores = model_score_table(price_baseline_predictions)
print(price_baseline_scores[price_baseline_scores["area"].eq("ALL")].sort_values("mae"))

weather_baseline_predictions = run_baseline_comparator(
    panel,
    origins=weather_outer_origins,
    min_train_days=365,
)
weather_baseline_scores = model_score_table(weather_baseline_predictions)
print(weather_baseline_scores[weather_baseline_scores["area"].eq("ALL")].sort_values("mae"))

,model_label,area,rows,evaluated_rows,mae,rmse,bias,coverage,interval_width,missing_rate
0,median_weekday_exp_hl4_floor10_42d__median_wee...,ALL,232272,232272,151.269247,312.302673,4.405034,NaN,NaN,0.0
1,rolling_median_hour_weekend_56d,ALL,232272,232272,169.965432,357.931495,-0.658050,NaN,NaN,0.0
2,same_hour_last_week,ALL,232272,232272,189.401410,383.548085,-0.943815,NaN,NaN,0.0


## 4. Policy Grid Helpers

The policy grid is intentionally not a full Cartesian product. It compares a small set of interpretable regimes: fast versus monthly refitting, no recency bias versus moderate versus aggressive recency bias, and full-history versus hard one-year memory. Feature sets and target modes remain part of the candidate grid inside each policy.


In [6]:
from dkenergy_forecast.features import tabular_feature_columns_for_set


def policy_metadata(policy, *, family):
    recency = policy["recency"]
    return {
        "family": family,
        "policy_label": policy["policy_label"],
        "retune_every_months": policy["retune_every_months"],
        "retrain_every_days": policy["retrain_every_days"],
        "training_origin_days": policy["training_origin_days"],
        "policy_recency_label": recency.label,
        "policy_sample_weight_half_life_days": recency.half_life_days,
        "policy_sample_weight_floor": recency.floor,
        "policy_rationale": policy["rationale"],
    }


def policy_model_prefix(family, policy):
    return f"{family}_catboost__{policy['policy_label']}"


def policy_config(policy, *, family, backtest_months):
    return CatBoostValidationConfig(
        **BASE_TUNING_CONFIG,
        validation_months=backtest_months,
        retune_every_months=policy["retune_every_months"],
        retrain_every_days=policy["retrain_every_days"],
        training_origin_days=policy["training_origin_days"],
        model_prefix=policy_model_prefix(family, policy),
    )


def policy_candidates(feature_sets, policy):
    return build_candidate_grid(
        feature_sets=feature_sets,
        target_modes=TARGET_MODES,
        recency_specs=[policy["recency"]],
        search_profiles=SEARCH_PROFILES,
    )


def annotate_frame(frame, metadata):
    output = frame.copy()
    for key, value in metadata.items():
        output[key] = value
    return output


def annotate_result(result, policy, *, family):
    metadata = policy_metadata(policy, family=family)
    annotated_frames = {
        attribute: annotate_frame(getattr(result, attribute), metadata)
        for attribute in [
            "candidate_tuning_scores",
            "selected_validation_configs",
            "catboost_predictions",
            "catboost_replay_metadata",
            "feature_importance",
            "combined_model_scores",
            "outer_month_model_scores",
            "per_origin_model_scores",
            "per_origin_deltas",
        ]
    }
    return replace(result, **annotated_frames)


def run_policy_grid(*, family, feature_sets, output_dir, experiment_frame, baseline_predictions, backtest_months, policy_grid):
    if not RUN_FULL_TUNING:
        raise RuntimeError("RUN_FULL_TUNING is False. This notebook is configured to run full tuning by default.")

    results = {}
    for policy in policy_grid:
        policy_label = policy["policy_label"]
        candidates = policy_candidates(feature_sets, policy)
        config = policy_config(policy, family=family, backtest_months=backtest_months)
        print(f"=== {family} policy {policy_label}: {len(candidates)} candidate(s) ===")
        result = run_catboost_validation(
            experiment_frame,
            candidates=candidates,
            config=config,
            baseline_predictions=baseline_predictions,
            progress=print,
        )
        result = annotate_result(result, policy, family=family)
        write_catboost_validation_artifacts(
            output_dir / policy_label,
            result,
            manifest={
                "run_id": f"notebook_{family}_catboost_policy_{policy_label}",
                "created_at_utc": datetime.now(timezone.utc).isoformat(),
                "backtest_months": backtest_months,
                "feature_sets": feature_sets,
                "target_modes": TARGET_MODES,
                "search_profiles": SEARCH_PROFILES,
                "policy": policy_metadata(policy, family=family),
                "config": {
                    **BASE_TUNING_CONFIG,
                    "validation_months": backtest_months,
                    "retune_every_months": policy["retune_every_months"],
                    "retrain_every_days": policy["retrain_every_days"],
                    "training_origin_days": policy["training_origin_days"],
                },
            },
        )
        results[policy_label] = result
    return results


def concat_result_frames(results, attribute):
    frames = [getattr(result, attribute) for result in results.values()]
    frames = [frame for frame in frames if not frame.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


feature_counts = []
for experiment, experiment_frame, feature_sets in [
    ("price", price_frame, PRICE_FEATURE_SETS),
    ("weather", weather_frame, WEATHER_FEATURE_SETS),
]:
    for feature_set in feature_sets:
        columns = tabular_feature_columns_for_set(experiment_frame, feature_set)
        feature_counts.append({
            "experiment": experiment,
            "feature_set": feature_set,
            "feature_count": len(columns),
            "weather_feature_count": sum(column.startswith("weather_") for column in columns),
        })
pd.DataFrame(feature_counts).sort_values(["experiment", "weather_feature_count", "feature_set"])


,experiment,feature_set,feature_count,weather_feature_count
1,price,price_baseline_calendar,8,0
0,price,price_full_engineered,21,0
2,price,price_lags_calendar,12,0
3,weather,price_full_engineered,21,0
5,weather,all_weather,55,34
4,weather,ensemble,77,56
6,weather,all_weather_plus_ensemble,111,90


In [ ]:
price_results = run_policy_grid(
    family="price",
    feature_sets=PRICE_FEATURE_SETS,
    output_dir=PRICE_RESULT_DIR,
    experiment_frame=price_frame,
    baseline_predictions=price_baseline_predictions,
    backtest_months=PRICE_BACKTEST_MONTHS,
    policy_grid=PRICE_POLICY_GRID,
)


=== price policy q3_r7_full_hl180: 6 candidate(s) ===


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/lib/python3.10/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-07-05 16:51:41,635] A new study created in memory with name: no-name-f6605f5b-9f75-4128-82a3-0a071c55a247


Retune 2013-04..2013-06: 91 validation origin(s), retrain every 7 day(s).
  Tuning price_full_engineered__direct__conservative__exp_hl180 on 2013-01-01..2013-04-01 (21 features)...


[I 2026-07-05 16:51:41,868] Trial 0 finished with value: 50.021169715934874 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 700, 'depth': 3, 'learning_rate': 0.013831748681407519, 'l2_leaf_reg': 11.900590783184242, 'random_strength': 17.591170623948834, 'border_count': 64, 'boosting_type': 'Plain', 'leaf_estimation_iterations': 1, 'subsample': 0.7045474901621301, 'rsm': 0.6641915784487018}. Best is trial 0 with value: 50.021169715934874.
[I 2026-07-05 16:51:42,347] Trial 1 finished with value: 46.05859524932997 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 500, 'depth': 5, 'learning_rate': 0.013365201439497277, 'l2_leaf_reg': 23.99324290681271, 'random_strength': 8.594513179286452, 'border_count': 64, 'boosting_type': 'Ordered', 'leaf_estimation_iterations': 1, 'subsample': 0.8322634555704315, 'rsm': 0.6596834432905521}. Best is trial 1 with value: 46.05859524932997.
[I 2026-07-05 16:51:42,675] Trial 2 finished with value: 51.2905501766498 and parameters: 

  Tuning price_full_engineered__residual_baseline__conservative__exp_hl180 on 2013-01-01..2013-04-01 (21 features)...


...

[I 2026-07-05 17:39:11,369] Trial 0 finished with value: 43.1779276832104 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 700, 'depth': 3, 'learning_rate': 0.013831748681407519, 'l2_leaf_reg': 11.900590783184242, 'random_strength': 17.591170623948834, 'border_count': 64, 'boosting_type': 'Plain', 'leaf_estimation_iterations': 1, 'subsample': 0.7045474901621301, 'rsm': 0.6641915784487018}. Best is trial 0 with value: 43.1779276832104.
[I 2026-07-05 17:39:14,691] Trial 1 finished with value: 41.8718168756348 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 500, 'depth': 5, 'learning_rate': 0.013365201439497277, 'l2_leaf_reg': 23.99324290681271, 'random_strength': 8.594513179286452, 'border_count': 64, 'boosting_type': 'Ordered', 'leaf_estimation_iterations': 1, 'subsample': 0.8322634555704315, 'rsm': 0.6596834432905521}. Best is trial 1 with value: 41.8718168756348.
[I 2026-07-05 17:39:17,365] Trial 2 finished with value: 41.57805757147594 and parameters: {'boo

  Tuning price_lags_calendar__direct__conservative__exp_hl180 on 2016-01-01..2016-04-01 (12 features)...


[I 2026-07-05 17:39:29,571] Trial 0 finished with value: 42.24865422320851 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 700, 'depth': 3, 'learning_rate': 0.013831748681407519, 'l2_leaf_reg': 11.900590783184242, 'random_strength': 17.591170623948834, 'border_count': 64, 'boosting_type': 'Plain', 'leaf_estimation_iterations': 1, 'subsample': 0.7045474901621301, 'rsm': 0.6641915784487018}. Best is trial 0 with value: 42.24865422320851.
[I 2026-07-05 17:39:33,279] Trial 1 finished with value: 39.52758844081509 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 500, 'depth': 5, 'learning_rate': 0.013365201439497277, 'l2_leaf_reg': 23.99324290681271, 'random_strength': 8.594513179286452, 'border_count': 64, 'boosting_type': 'Ordered', 'leaf_estimation_iterations': 1, 'subsample': 0.8322634555704315, 'rsm': 0.6596834432905521}. Best is trial 1 with value: 39.52758844081509.
[I 2026-07-05 17:39:37,958] Trial 2 finished with value: 41.606657405410026 and parameters: 

  Tuning price_lags_calendar__residual_baseline__conservative__exp_hl180 on 2016-01-01..2016-04-01 (12 features)...


[I 2026-07-05 17:39:56,426] Trial 0 finished with value: 44.72239529558799 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 700, 'depth': 3, 'learning_rate': 0.013831748681407519, 'l2_leaf_reg': 11.900590783184242, 'random_strength': 17.591170623948834, 'border_count': 64, 'boosting_type': 'Plain', 'leaf_estimation_iterations': 1, 'subsample': 0.7045474901621301, 'rsm': 0.6641915784487018}. Best is trial 0 with value: 44.72239529558799.
[I 2026-07-05 17:40:00,103] Trial 1 finished with value: 44.029176289925665 and parameters: {'bootstrap_type': 'Bernoulli', 'iterations': 500, 'depth': 5, 'learning_rate': 0.013365201439497277, 'l2_leaf_reg': 23.99324290681271, 'random_strength': 8.594513179286452, 'border_count': 64, 'boosting_type': 'Ordered', 'leaf_estimation_iterations': 1, 'subsample': 0.8322634555704315, 'rsm': 0.6596834432905521}. Best is trial 1 with value: 44.029176289925665.
[I 2026-07-05 17:40:04,926] Trial 2 finished with value: 44.955963974712255 and parameters

### Price-Only Policy Diagnostics

For each policy, the selected path is the candidate chosen by inner tuning for each quarterly block. This means a policy can select different feature sets or target modes as it rolls forward; the policy score reflects that whole adaptive path.


In [ ]:
def selected_policy_predictions(results, *, family):
    frames = []
    for policy_label, result in results.items():
        selected = result.catboost_predictions[result.catboost_predictions["selected_by_tuning"]].copy()
        selected["candidate_model_label"] = selected["model_label"]
        selected["model_label"] = f"{family}_selected__{policy_label}"
        frames.append(selected)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def selected_policy_delta_table(results):
    frames = []
    for result in results.values():
        selected = result.per_origin_deltas[result.per_origin_deltas["selected_by_tuning"]].copy()
        frames.append(selected)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def selected_config_table(results):
    return concat_result_frames(results, "selected_validation_configs").sort_values(
        ["policy_label", "retune_at_utc", "candidate_label"]
    ).reset_index(drop=True)


def policy_score_table(results, *, family, baseline_predictions, policy_grid, backtest_months):
    selected = selected_policy_predictions(results, family=family)
    combined = pd.concat([baseline_predictions, selected], ignore_index=True)
    scores = model_score_table(combined)
    policy_scores = scores[
        scores["area"].eq("ALL")
        & scores["model_label"].str.startswith(f"{family}_selected__")
    ].copy()
    policy_scores["policy_label"] = policy_scores["model_label"].str.removeprefix(f"{family}_selected__")
    policy_scores["family"] = family
    policy_scores["backtest_start_month"] = backtest_months[0]
    policy_scores["backtest_end_month"] = backtest_months[-1]
    policy_scores["backtest_month_count"] = len(backtest_months)

    best_baseline = (
        scores[scores["area"].eq("ALL") & ~scores["model_label"].str.startswith(f"{family}_selected__")]
        .sort_values(["mae", "model_label"])
        .iloc[0]
    )
    policy_scores["best_baseline_label"] = best_baseline["model_label"]
    policy_scores["best_baseline_mae"] = float(best_baseline["mae"])
    policy_scores["mae_minus_best_baseline"] = policy_scores["mae"] - policy_scores["best_baseline_mae"]

    metadata = pd.DataFrame([policy_metadata(policy, family=family) for policy in policy_grid])
    return (
        policy_scores.merge(metadata, on=["family", "policy_label"], how="left")
        .sort_values(["mae", "policy_label"])
        .reset_index(drop=True)
    )


def policy_month_score_table(results, *, family, baseline_predictions):
    selected = selected_policy_predictions(results, family=family)
    combined = pd.concat([baseline_predictions, selected], ignore_index=True)
    scores = outer_month_score_table(combined)
    selected_scores = scores[
        scores["model_label"].str.startswith(f"{family}_selected__")
    ].copy()
    selected_scores["policy_label"] = selected_scores["model_label"].str.removeprefix(f"{family}_selected__")
    return selected_scores.sort_values(["policy_label", "outer_month"]).reset_index(drop=True)


price_policy_scores = policy_score_table(
    price_results,
    family="price",
    baseline_predictions=price_baseline_predictions,
    policy_grid=PRICE_POLICY_GRID,
    backtest_months=PRICE_BACKTEST_MONTHS,
)
price_policy_scores


In [ ]:
price_selected_configs = selected_config_table(price_results)
price_selected_configs[[
    "policy_label",
    "validation_block",
    "candidate_label",
    "feature_set",
    "target_mode",
    "recency_label",
    "tuning_mae",
    "retrain_every_days",
    "training_origin_days",
]].sort_values(["policy_label", "validation_block"])


In [ ]:
selected_price_deltas = selected_policy_delta_table(price_results)
selected_price_delta_summary = selected_price_deltas.groupby(["policy_label", "validation_block"]).agg(
    origins=("forecast_origin_utc", "nunique"),
    mean_delta=("catboost_minus_best_baseline_mae", "mean"),
    median_delta=("catboost_minus_best_baseline_mae", "median"),
    win_rate=("catboost_minus_best_baseline_mae", lambda s: (s < 0).mean()),
    mean_validation_minus_tuning=("validation_minus_tuning_mae", "mean"),
).reset_index()
selected_price_delta_summary.sort_values(["policy_label", "validation_block"])


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(
    data=selected_price_deltas,
    x="policy_label",
    y="catboost_minus_best_baseline_mae",
    hue="validation_block",
    ax=ax,
)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Price CatBoost policies: selected-path per-origin MAE delta vs best baseline")
ax.set_ylabel("CatBoost MAE - best baseline MAE")
ax.set_xlabel("Policy")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


<p style="color:#b00020"><strong>Interpretation.</strong> The relevant comparison is whether an adaptation policy stays below the baseline across several future blocks, not whether one candidate wins one convenient split. Watch for policies that win only in one quarter, policies where outer MAE is much worse than the inner tuning MAE, and policies that oscillate between direct and residual formulations.</p>


## 5. Weather-Enhanced Policy Grid

Weather features enter after the price-only policy story is visible. The weather grid uses the same policies and includes `price_full_engineered` as a no-weather anchor, so a weather policy can reveal when weather features are genuinely worth selecting and when the price-only signal is enough.


In [ ]:
weather_results = run_policy_grid(
    family="weather",
    feature_sets=WEATHER_FEATURE_SETS,
    output_dir=WEATHER_RESULT_DIR,
    experiment_frame=weather_frame,
    baseline_predictions=weather_baseline_predictions,
    backtest_months=WEATHER_BACKTEST_MONTHS,
    policy_grid=WEATHER_POLICY_GRID,
)


In [ ]:
weather_policy_scores = policy_score_table(
    weather_results,
    family="weather",
    baseline_predictions=weather_baseline_predictions,
    policy_grid=WEATHER_POLICY_GRID,
    backtest_months=WEATHER_BACKTEST_MONTHS,
)
weather_policy_scores


In [ ]:
weather_selected_configs = selected_config_table(weather_results)
weather_selected_configs[[
    "policy_label",
    "validation_block",
    "candidate_label",
    "feature_set",
    "target_mode",
    "recency_label",
    "tuning_mae",
    "retrain_every_days",
    "training_origin_days",
]].sort_values(["policy_label", "validation_block"])


### Weather Policy Diagnostics

A weather policy earns its keep only if the selected path improves future-block accuracy and actually chooses weather-conditioned feature sets often enough to justify the added data dependency.


In [ ]:
weather_feature_adoption = (
    weather_selected_configs.assign(uses_weather=lambda df: df["feature_set"].ne("price_full_engineered"))
    .groupby("policy_label")
    .agg(
        validation_blocks=("validation_block", "nunique"),
        weather_selected_blocks=("uses_weather", "sum"),
        weather_selection_rate=("uses_weather", "mean"),
        selected_feature_sets=("feature_set", lambda values: ", ".join(sorted(set(values)))),
    )
    .reset_index()
)
weather_feature_adoption


In [ ]:
selected_weather_deltas = selected_policy_delta_table(weather_results)
selected_weather_delta_summary = selected_weather_deltas.groupby(["policy_label", "validation_block"]).agg(
    origins=("forecast_origin_utc", "nunique"),
    mean_delta=("catboost_minus_best_baseline_mae", "mean"),
    median_delta=("catboost_minus_best_baseline_mae", "median"),
    win_rate=("catboost_minus_best_baseline_mae", lambda s: (s < 0).mean()),
    mean_validation_minus_tuning=("validation_minus_tuning_mae", "mean"),
).reset_index()
selected_weather_delta_summary.sort_values(["policy_label", "validation_block"])


In [ ]:
combined_policy_scores = pd.concat(
    [
        price_policy_scores.assign(experiment_family="price-only"),
        weather_policy_scores.assign(experiment_family="weather-grid"),
    ],
    ignore_index=True,
)
combined_policy_scores[[
    "experiment_family",
    "policy_label",
    "backtest_start_month",
    "backtest_end_month",
    "backtest_month_count",
    "model_label",
    "mae",
    "rmse",
    "bias",
    "best_baseline_label",
    "best_baseline_mae",
    "mae_minus_best_baseline",
    "retune_every_months",
    "retrain_every_days",
    "training_origin_days",
    "policy_recency_label",
]].sort_values("mae")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_frame = pd.concat([
    selected_price_deltas.assign(experiment_family="price-only"),
    selected_weather_deltas.assign(experiment_family="weather-grid"),
], ignore_index=True)
sns.boxplot(
    data=plot_frame,
    x="policy_label",
    y="catboost_minus_best_baseline_mae",
    hue="experiment_family",
    ax=ax,
)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Selected CatBoost policy paths vs best baseline")
ax.set_ylabel("CatBoost MAE - best baseline MAE")
ax.set_xlabel("Policy")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
selected_weather_keys = weather_selected_configs[["policy_label", "candidate_label"]].drop_duplicates()
weather_importance = concat_result_frames(weather_results, "feature_importance")
selected_weather_importance = weather_importance.merge(
    selected_weather_keys,
    on=["policy_label", "candidate_label"],
    how="inner",
)
top_importance = (
    selected_weather_importance.groupby("feature", as_index=False)["importance"]
    .mean()
    .sort_values("importance", ascending=False)
    .head(25)
)
top_importance


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=top_importance, y="feature", x="importance", ax=ax, color="#54a24b")
ax.set_title("Top mean feature importances for selected weather-policy candidates")
ax.set_xlabel("Mean CatBoost importance")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


<p style="color:#b00020"><strong>Interpretation.</strong> Weather is useful only if it improves the selected policy path across future blocks. If the weather grid mostly selects `price_full_engineered`, the policy is telling us that the extra data dependency is not consistently worth it under the current feature construction.</p>


## 6. Policy Selection Summary

This final section records exploratory winners at the policy level. Price-only policies are selected in the long EDS-price arena, while weather policies are selected in the shorter weather-overlap arena. Treat the two winners as within-arena choices, not as a direct apples-to-apples promotion gate; the output is meant to narrow the next experiment and identify the adaptation regime that deserves an honest holdout or live shadow run.


In [ ]:
def best_policy_row(policy_scores, *, experiment_family):
    best = policy_scores.sort_values(["mae", "policy_label"]).iloc[0].to_dict()
    best["experiment_family"] = experiment_family
    best["selection_rule"] = "lowest_selected_policy_validation_mae"
    return best


final_policy_selections = pd.DataFrame([
    best_policy_row(price_policy_scores, experiment_family="price-only"),
    best_policy_row(weather_policy_scores, experiment_family="weather-grid"),
])[[
    "experiment_family",
    "policy_label",
    "backtest_start_month",
    "backtest_end_month",
    "backtest_month_count",
    "model_label",
    "mae",
    "rmse",
    "bias",
    "best_baseline_label",
    "best_baseline_mae",
    "mae_minus_best_baseline",
    "retune_every_months",
    "retrain_every_days",
    "training_origin_days",
    "policy_recency_label",
    "policy_sample_weight_half_life_days",
    "policy_sample_weight_floor",
    "selection_rule",
]]

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
price_policy_scores.to_parquet(RESULT_ROOT / "price_policy_scores.parquet", index=False)
weather_policy_scores.to_parquet(RESULT_ROOT / "weather_policy_scores.parquet", index=False)
combined_policy_scores.to_parquet(RESULT_ROOT / "combined_policy_scores.parquet", index=False)
price_selected_configs.to_parquet(RESULT_ROOT / "price_selected_policy_configs.parquet", index=False)
weather_selected_configs.to_parquet(RESULT_ROOT / "weather_selected_policy_configs.parquet", index=False)
final_policy_selections.to_parquet(RESULT_ROOT / "final_policy_selections.parquet", index=False)
final_policy_selections


In [ ]:
def chosen_policy_month_comparison(results, *, family, policy_label, final_label, baseline_predictions):
    result = results[policy_label]
    chosen = result.catboost_predictions[result.catboost_predictions["selected_by_tuning"]].copy()
    chosen["model_label"] = final_label
    comparison = outer_month_score_table(pd.concat([baseline_predictions, chosen], ignore_index=True))
    best_baseline_by_month = (
        comparison[~comparison["model_label"].eq(final_label)]
        .sort_values(["outer_month", "mae", "model_label"])
        .groupby("outer_month", as_index=False)
        .head(1)
        [["outer_month", "model_label", "mae"]]
        .rename(columns={"model_label": "best_baseline_label", "mae": "best_baseline_mae"})
    )
    chosen_by_month = comparison[comparison["model_label"].eq(final_label)].merge(
        best_baseline_by_month,
        on="outer_month",
        how="left",
    )
    chosen_by_month["family"] = family
    chosen_by_month["policy_label"] = policy_label
    chosen_by_month["mae_minus_best_baseline"] = chosen_by_month["mae"] - chosen_by_month["best_baseline_mae"]
    return chosen_by_month

price_best_policy_label = final_policy_selections.loc[
    final_policy_selections["experiment_family"].eq("price-only"),
    "policy_label",
].iloc[0]
weather_best_policy_label = final_policy_selections.loc[
    final_policy_selections["experiment_family"].eq("weather-grid"),
    "policy_label",
].iloc[0]

final_policy_month_comparison = pd.concat(
    [
        chosen_policy_month_comparison(
            price_results,
            family="price-only",
            policy_label=price_best_policy_label,
            final_label="best_price_catboost_policy",
            baseline_predictions=price_baseline_predictions,
        ),
        chosen_policy_month_comparison(
            weather_results,
            family="weather-grid",
            policy_label=weather_best_policy_label,
            final_label="best_weather_catboost_policy",
            baseline_predictions=weather_baseline_predictions,
        ),
    ],
    ignore_index=True,
)
final_policy_month_comparison.to_parquet(
    RESULT_ROOT / "final_policy_month_comparison.parquet",
    index=False,
)
final_policy_month_comparison.sort_values(["family", "outer_month"])


<p style="color:#b00020"><strong>Final modeling read.</strong> The notebook now records a rolling adaptation-policy comparison rather than a static candidate comparison. A good policy should win or stay competitive across multiple future blocks, avoid a large gap between inner tuning MAE and outer replay MAE, and keep its added complexity explainable. Production promotion remains manual and source-controlled: update the fixed CatBoost config/registry, run a smoke publish, and commit the result; notebook winners do not auto-promote.</p>
